In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
import re
from datetime import datetime
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

# =========================
# SETTINGS
# =========================

input_file = "goobjoog_ciyaaraha_links.xlsx"

# Use a NEW sample output file first
output_file = "goobjoog_ciyaaraha_scraped_articles_SAMPLE10.xlsx"

TEST_LIMIT = 10          # First test only 10 links
SAVE_EVERY = 2           # Save checkpoint after every 2 articles
MIN_BODY_WORDS = 50      # Used only for warning/checking
REQUEST_DELAY = 2.5

CATEGORY = "ciyaaro"
SOURCE = "Goobjoog"

FORCE_RESCRAPE_EMPTY = True
# If output_file exists, this will re-scrape rows whose body is empty/too short


# =========================
# SESSION
# =========================

def get_robust_session():
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False
    )

    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "so,en-US;q=0.9,en;q=0.8",
        "Connection": "keep-alive",
    })

    return session


# =========================
# TEXT CLEANING HELPERS
# =========================

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\xa0", " ")
    text = text.replace("\ufeff", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


STOP_PATTERNS = [
    r"^W/D\s*[-–]",
    r"^W/Q\s*[-–]",
    r"^Isla Qaybtaan$",
    r"^Qaybaha$",
    r"^Mawduucyada$",
    r"^Noocyada Qormooyinka$",
    r"^Goobjoog$",
    r"^La Xiriir$",
    r"^©\s*\d{4}",
    r"^Xuquuqda faafintu",
]

REMOVE_PATTERNS = [
    r"^Goobjoog News$",
    r"^Googjoog News$",
    r"^Dhageyso$",
    r"^Halkaan hoose ka dhageyso",
    r"^Halkaan ka Akhriso",
    r"^Rukumo",
    r"^Share",
    r"^Facebook$",
    r"^Twitter$",
    r"^WhatsApp$",
    r"^Telegram$",
]


def should_stop_line(line):
    return any(re.search(pattern, line, flags=re.IGNORECASE) for pattern in STOP_PATTERNS)


def should_remove_line(line):
    return any(re.search(pattern, line, flags=re.IGNORECASE) for pattern in REMOVE_PATTERNS)


def clean_lines(lines):
    cleaned = []

    for raw_line in lines:
        line = normalize_text(raw_line)

        if not line:
            continue

        if should_stop_line(line):
            break

        if should_remove_line(line):
            continue

        # remove very tiny non-article fragments
        if len(line.split()) < 3:
            continue

        cleaned.append(line)

    # Remove repeated adjacent lines
    deduped = []
    for line in cleaned:
        if not deduped or deduped[-1] != line:
            deduped.append(line)

    return "\n".join(deduped).strip()


# =========================
# EXTRACTION HELPERS
# =========================

def extract_headline(soup):
    selectors = [
        "h1.entry-title",
        "h1.post-title",
        "h1"
    ]

    for selector in selectors:
        tag = soup.select_one(selector)
        if tag:
            headline = normalize_text(tag.get_text(" ", strip=True))
            if headline:
                return headline

    return ""


def extract_paragraphs_from_container(container):
    # Remove unwanted HTML blocks
    for tag in container.select("script, style, aside, nav, footer, header, form, iframe, ins"):
        tag.decompose()

    paragraphs = container.find_all("p")

    if paragraphs:
        return [p.get_text(" ", strip=True) for p in paragraphs]

    # fallback if the container has text but not paragraph tags
    return container.get_text("\n", strip=True).splitlines()


def extract_body(soup):
    # First try likely article containers
    container_selectors = [
        "div.entry-content",
        "article div.entry-content",
        "div.post-content",
        "div.article-content",
        "div.td-post-content",
        "article",
        "main",
    ]

    best_body = ""

    for selector in container_selectors:
        container = soup.select_one(selector)

        if not container:
            continue

        lines = extract_paragraphs_from_container(container)
        body = clean_lines(lines)

        if len(body.split()) > len(best_body.split()):
            best_body = body

        # good enough, return early
        if len(body.split()) >= MIN_BODY_WORDS:
            return body

    # Final fallback: collect all page paragraphs
    all_paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    fallback_body = clean_lines(all_paragraphs)

    if len(fallback_body.split()) > len(best_body.split()):
        best_body = fallback_body

    return best_body


def scrape_goobjoog_article(url, session):
    try:
        response = session.get(url, timeout=25)

        status_code = response.status_code

        if status_code != 200:
            return {
                "headline": "",
                "body": "",
                "status": f"failed_status_{status_code}",
                "error": f"HTTP status {status_code}"
            }

        soup = BeautifulSoup(response.content, "html.parser")

        headline = extract_headline(soup)
        body = extract_body(soup)

        body_word_count = len(body.split())

        if not headline and not body:
            status = "failed_no_headline_no_body"
        elif not body:
            status = "warning_no_body"
        elif body_word_count < MIN_BODY_WORDS:
            status = "warning_short_body"
        else:
            status = "ok"

        return {
            "headline": headline,
            "body": body,
            "status": status,
            "error": ""
        }

    except Exception as e:
        return {
            "headline": "",
            "body": "",
            "status": "exception",
            "error": str(e)
        }


# =========================
# LOAD LINKS
# =========================

if not os.path.exists(input_file):
    raise FileNotFoundError(f"{input_file} not found. Make sure it is in the same folder as this notebook.")

df_links = pd.read_excel(input_file)

print("Columns in links file:")
print(df_links.columns.tolist())

# Accept URL or url column
if "URL" in df_links.columns:
    url_col = "URL"
elif "url" in df_links.columns:
    url_col = "url"
else:
    raise ValueError("No URL column found. Your file must contain 'URL' or 'url'.")

urls_to_scrape = (
    df_links[url_col]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

if TEST_LIMIT is not None:
    urls_to_scrape = urls_to_scrape[:TEST_LIMIT]

print(f"\nTotal URLs selected for this run: {len(urls_to_scrape)}")


# =========================
# CHECKPOINT / RESUME
# =========================

scraped_data = []
processed_urls = set()

if os.path.exists(output_file):
    df_existing = pd.read_excel(output_file)

    if FORCE_RESCRAPE_EMPTY and not df_existing.empty:
        df_existing["body"] = df_existing["body"].fillna("").astype(str)
        df_existing["body_word_count"] = df_existing["body"].str.split().str.len()

        df_good_existing = df_existing[df_existing["body_word_count"] >= MIN_BODY_WORDS].copy()

        removed_count = len(df_existing) - len(df_good_existing)

        scraped_data = df_good_existing.drop(columns=["body_word_count"], errors="ignore").to_dict("records")
        processed_urls = set(df_good_existing["url"].astype(str).tolist())

        print(f"\nResuming from checkpoint: {len(processed_urls)} good rows kept.")
        print(f"Rows with empty/short body will be re-scraped: {removed_count}")
    else:
        scraped_data = df_existing.to_dict("records")
        processed_urls = set(df_existing["url"].astype(str).tolist())
        print(f"\nResuming from checkpoint: {len(processed_urls)} rows already scraped.")


# =========================
# MAIN SCRAPE LOOP
# =========================

session = get_robust_session()

print("\nStarting Goobjoog Ciyaaro sample scrape...\n")

for i, url in enumerate(urls_to_scrape, start=1):
    if url in processed_urls:
        print(f"[{i}/{len(urls_to_scrape)}] Skipping already processed: {url}")
        continue

    print("=" * 90)
    print(f"[{i}/{len(urls_to_scrape)}] Scraping: {url}")

    result = scrape_goobjoog_article(url, session)

    headline = result["headline"]
    body = result["body"]
    status = result["status"]
    error = result["error"]

    body_word_count = len(body.split())

    row = {
        "url": url,
        "headline": headline,
        "body": body,
        "category": CATEGORY,
        "source": SOURCE,
        "status": status,
        "body_word_count": body_word_count,
        "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "error": error
    }

    scraped_data.append(row)
    processed_urls.add(url)

    print(f"Status: {status}")
    print(f"Headline: {headline[:120] if headline else 'NO HEADLINE'}")
    print(f"Body words: {body_word_count}")
    print(f"Body preview: {body[:250].replace(chr(10), ' ') if body else 'NO BODY'}")

    if len(scraped_data) % SAVE_EVERY == 0:
        pd.DataFrame(scraped_data).to_excel(output_file, index=False)
        print(f"Checkpoint saved: {len(scraped_data)} rows -> {output_file}")

    time.sleep(REQUEST_DELAY)

# Final save
df_out = pd.DataFrame(scraped_data)
df_out.to_excel(output_file, index=False)

print("\n" + "=" * 90)
print(f"Scraping complete. Saved {len(df_out)} rows to {output_file}")
print("\nStatus counts:")
print(df_out["status"].value_counts(dropna=False))

print("\nBody word count summary:")
print(df_out["body_word_count"].describe())

df_out[["url", "headline", "body_word_count", "status", "body"]].head(10)

Columns in links file:
['URL']

Total URLs selected for this run: 10

Starting Goobjoog Ciyaaro sample scrape...

[1/10] Scraping: https://goobjoog.com/2020/12/31/yaa-guulaysan-doono-gobolka-banaadir-iyo-galmudug-tartanka-maamul-goboleedyada-2020/
Status: ok
Headline: Yaa Guulaysan Doono Gobolka Banaadir Iyo Galmudug Tartanka Maamul Goboleedyada 2020
Body words: 221
Body preview: Kooxaha Kubbadda Cagta Gobolka Banaadir iyo Galmudug ayaa markii ugu horeesay wada dheeli doono heerka ugu danbeeya tartanka kubbadda cagta Maamul Goboleedyada iyo Gobolka Banaadir. Labada koox oo ka soo wada baxay Groupka A,ayaa soo tiigsaday fiinal
[2/10] Scraping: https://goobjoog.com/2020/10/25/xildhibaan-shaacir-guul-ayaan-u-rajaynaayaa-wasiir-xamsa-iyo-wasaarada-ciyaaraha-dalka/
Status: ok
Headline: Xildhibaan Shaacir "Guul Ayaan U Rajaynaayaa Wasiir Xamsa Iyo Wasaarada Ciyaaraha Dalka"
Body words: 179
Body preview: Xildhibaan Cabdullahi Maxamed Aadan (Shaacir) oo ka tirsan Xildhibaanada Golaha shacabka 

,url,headline,body_word_count,status,body
0,https://goobjoog.com/2020/12/31/yaa-guulaysan-...,Yaa Guulaysan Doono Gobolka Banaadir Iyo Galmu...,221,ok,Kooxaha Kubbadda Cagta Gobolka Banaadir iyo Ga...
1,https://goobjoog.com/2020/10/25/xildhibaan-sha...,"Xildhibaan Shaacir ""Guul Ayaan U Rajaynaayaa W...",179,ok,Xildhibaan Cabdullahi Maxamed Aadan (Shaacir) ...
2,https://goobjoog.com/2024/01/21/akhriso-tartan...,Akhriso: Tartanka Dowlad Goboleedyada oo Dib u...,319,ok,Garoonka Kubadda Cagta ee Stadium Muqdisho wax...
3,https://goobjoog.com/2020/07/11/maxamuud-maxam...,"Maxamuud Maxamed ""Waxaan Diyaar U Nahay Horoma...",196,ok,Maamulka Ciyaaraha Degmada Diinsoor ayaa tarta...
4,https://goobjoog.com/2019/06/10/wax-badan-ka-o...,Wax Badan Ka Ogaw Kulanka Horseed VS Muqdisho ...,309,ok,Kooxda Ciidanka xooda dalka ee Horseed ayaa wa...
5,https://goobjoog.com/2020/06/16/ibraahim-geedo...,"Ibraahim Geedow ""Xiriirka Kubbbadda CagtaS Soo...",57,ok,Ibraahim Geedoow Cawaale oo ah qoraa iyo falan...
6,https://goobjoog.com/2020/08/29/daadir-amiin-w...,"Daadir Amiin ""Waan Ku Faraxsanahay Hanashada H...",107,ok,Kooxda Geeska Afrika ayaa ku guulaystay horyaa...
7,https://goobjoog.com/2019/03/28/hordhaca-horya...,Hordhaca Horyaalka Somaaliya-Yaa Guulo Badan K...,514,ok,Kooxda ciidan booliska Soomaaliyeed ee Heegan ...
8,https://goobjoog.com/2019/11/21/daaru-tarbiya-...,Daaru Tarbiya Oo Ku Guulaystay Tartanka Iskuulada,225,ok,Waxaa lasoo gaba-gabeeyay koobkii kubbadda cag...
9,https://goobjoog.com/2019/09/28/guddoomiye-cab...,"Guddoomiye Cabdiqani Saciid ""Xulka Dhallinyara...",334,ok,Xiriirka kubbadda cagta Bariga iyo Bartamaha A...


In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
from datetime import datetime
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

# =========================
# SETTINGS
# =========================

input_file = "goobjoog_ciyaaraha_links.xlsx"
output_file = "goobjoog_ciyaaraha_scraped_SAMPLE10.xlsx"

TEST_LIMIT = 10
SAVE_EVERY = 2
REQUEST_DELAY = 2.5

CATEGORY = "ciyaaro"
SOURCE = "Goobjoog"


# =========================
# SESSION
# =========================

def get_robust_session():
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False
    )

    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "so,en-US;q=0.9,en;q=0.8",
        "Connection": "keep-alive",
    })

    return session


# =========================
# SCRAPER FUNCTIONS
# =========================

def extract_headline(soup):
    headline_tag = (
        soup.select_one("h1.entry-title")
        or soup.select_one("h1.post-title")
        or soup.find("h1")
    )

    if headline_tag:
        return headline_tag.get_text(" ", strip=True)

    return ""


def extract_body(soup):
    """
    No cleaning here.
    We only try to find the article container and collect paragraph text.
    If no article container is found, we fall back to all <p> tags.
    """

    article_div = (
        soup.select_one("div.entry-content")
        or soup.select_one("article div.entry-content")
        or soup.select_one("div.post-content")
        or soup.select_one("div.article-content")
        or soup.select_one("div.td-post-content")
        or soup.find("article")
        or soup.find("main")
    )

    if article_div:
        paragraphs = article_div.find_all("p")
    else:
        paragraphs = soup.find_all("p")

    body = "\n".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
        if p.get_text(" ", strip=True)
    )

    return body


def scrape_goobjoog_article(url, session):
    try:
        response = session.get(url, timeout=25)

        if response.status_code != 200:
            return {
                "headline": "",
                "body": "",
                "status": f"failed_status_{response.status_code}",
                "error": f"HTTP status {response.status_code}"
            }

        soup = BeautifulSoup(response.content, "html.parser")

        headline = extract_headline(soup)
        body = extract_body(soup)

        if headline and body:
            status = "ok"
        elif headline and not body:
            status = "headline_only_no_body"
        elif body and not headline:
            status = "body_only_no_headline"
        else:
            status = "no_headline_no_body"

        return {
            "headline": headline,
            "body": body,
            "status": status,
            "error": ""
        }

    except Exception as e:
        return {
            "headline": "",
            "body": "",
            "status": "exception",
            "error": str(e)
        }


# =========================
# LOAD LINKS
# =========================

if not os.path.exists(input_file):
    raise FileNotFoundError(f"{input_file} not found. Put it in the same folder as this notebook/script.")

df_links = pd.read_excel(input_file)

print("Columns found in links file:")
print(df_links.columns.tolist())

if "URL" in df_links.columns:
    url_col = "URL"
elif "url" in df_links.columns:
    url_col = "url"
else:
    raise ValueError("No URL column found. Your file must contain either 'URL' or 'url'.")

urls_to_scrape = (
    df_links[url_col]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

urls_to_scrape = urls_to_scrape[:TEST_LIMIT]

print(f"\nSelected {len(urls_to_scrape)} URLs for sample scraping.")


# =========================
# CHECKPOINT / RESUME
# =========================

scraped_data = []
processed_urls = set()

if os.path.exists(output_file):
    df_existing = pd.read_excel(output_file)
    scraped_data = df_existing.to_dict("records")
    processed_urls = set(df_existing["url"].astype(str).tolist())

    print(f"\nResuming from checkpoint: {len(processed_urls)} URLs already scraped.")


# =========================
# MAIN LOOP
# =========================

session = get_robust_session()

print("\nStarting sample scrape...\n")

for i, url in enumerate(urls_to_scrape, start=1):
    if url in processed_urls:
        print(f"[{i}/{len(urls_to_scrape)}] Skipping already scraped: {url}")
        continue

    print("=" * 90)
    print(f"[{i}/{len(urls_to_scrape)}] Scraping: {url}")

    result = scrape_goobjoog_article(url, session)

    headline = result["headline"]
    body = result["body"]
    status = result["status"]
    error = result["error"]
    body_word_count = len(body.split()) if body else 0

    row = {
        "url": url,
        "headline": headline,
        "body": body,
        "category": CATEGORY,
        "source": SOURCE,
        "status": status,
        "body_word_count": body_word_count,
        "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "error": error
    }

    scraped_data.append(row)
    processed_urls.add(url)

    print(f"Status: {status}")
    print(f"Headline: {headline[:150] if headline else 'NO HEADLINE'}")
    print(f"Body words: {body_word_count}")
    print(f"Body preview: {body[:300].replace(chr(10), ' ') if body else 'NO BODY'}")

    if len(scraped_data) % SAVE_EVERY == 0:
        pd.DataFrame(scraped_data).to_excel(output_file, index=False)
        print(f"Checkpoint saved: {len(scraped_data)} rows -> {output_file}")

    time.sleep(REQUEST_DELAY)

# Final save
df_out = pd.DataFrame(scraped_data)
df_out.to_excel(output_file, index=False)

print("\n" + "=" * 90)
print(f"Sample scraping complete. Saved {len(df_out)} rows to {output_file}")

print("\nStatus counts:")
print(df_out["status"].value_counts(dropna=False))

print("\nBody word count summary:")
print(df_out["body_word_count"].describe())

df_out[["url", "headline", "body_word_count", "status", "body"]].head(10)

Columns found in links file:
['URL']

Selected 10 URLs for sample scraping.

Starting sample scrape...

[1/10] Scraping: https://goobjoog.com/2020/12/31/yaa-guulaysan-doono-gobolka-banaadir-iyo-galmudug-tartanka-maamul-goboleedyada-2020/
Status: ok
Headline: Yaa Guulaysan Doono Gobolka Banaadir Iyo Galmudug Tartanka Maamul Goboleedyada 2020
Body words: 224
Body preview: Kooxaha Kubbadda Cagta Gobolka Banaadir iyo Galmudug ayaa markii ugu horeesay wada dheeli doono heerka ugu danbeeya tartanka kubbadda cagta Maamul Goboleedyada iyo Gobolka Banaadir. Labada koox oo ka soo wada baxay Groupka A,ayaa soo tiigsaday fiinalaha tartan markii kowaad,tan iyo markii labilaaway
[2/10] Scraping: https://goobjoog.com/2020/10/25/xildhibaan-shaacir-guul-ayaan-u-rajaynaayaa-wasiir-xamsa-iyo-wasaarada-ciyaaraha-dalka/
Status: ok
Headline: Xildhibaan Shaacir “Guul Ayaan U Rajaynaayaa Wasiir Xamsa Iyo Wasaarada Ciyaaraha Dalka”
Body words: 179
Body preview: Xildhibaan Cabdullahi Maxamed Aadan (Shaacir) oo 

,url,headline,body_word_count,status,body
0,https://goobjoog.com/2020/12/31/yaa-guulaysan-...,Yaa Guulaysan Doono Gobolka Banaadir Iyo Galmu...,224,ok,Kooxaha Kubbadda Cagta Gobolka Banaadir iyo Ga...
1,https://goobjoog.com/2020/10/25/xildhibaan-sha...,Xildhibaan Shaacir “Guul Ayaan U Rajaynaayaa W...,179,ok,Xildhibaan Cabdullahi Maxamed Aadan (Shaacir) ...
2,https://goobjoog.com/2024/01/21/akhriso-tartan...,Akhriso: Tartanka Dowlad Goboleedyada oo Dib u...,319,ok,Garoonka Kubadda Cagta ee Stadium Muqdisho wax...
3,https://goobjoog.com/2020/07/11/maxamuud-maxam...,Maxamuud Maxamed “Waxaan Diyaar U Nahay Horoma...,199,ok,Maamulka Ciyaaraha Degmada Diinsoor ayaa tarta...
4,https://goobjoog.com/2019/06/10/wax-badan-ka-o...,Wax Badan Ka Ogaw Kulanka Horseed VS Muqdisho ...,320,ok,Kooxda Ciidanka xooda dalka ee Horseed ayaa wa...
5,https://goobjoog.com/2020/06/16/ibraahim-geedo...,Ibraahim Geedow “Xiriirka Kubbbadda CagtaS Soo...,57,ok,Ibraahim Geedoow Cawaale oo ah qoraa iyo falan...
6,https://goobjoog.com/2020/08/29/daadir-amiin-w...,Daadir Amiin “Waan Ku Faraxsanahay Hanashada H...,110,ok,Kooxda Geeska Afrika ayaa ku guulaystay horyaa...
7,https://goobjoog.com/2019/03/28/hordhaca-horya...,Hordhaca Horyaalka Somaaliya-Yaa Guulo Badan K...,523,ok,Kooxda ciidan booliska Soomaaliyeed ee Heegan ...
8,https://goobjoog.com/2019/11/21/daaru-tarbiya-...,Daaru Tarbiya Oo Ku Guulaystay Tartanka Iskuulada,225,ok,Waxaa lasoo gaba-gabeeyay koobkii kubbadda cag...
9,https://goobjoog.com/2019/09/28/guddoomiye-cab...,Guddoomiye Cabdiqani Saciid “Xulka Dhallinyara...,337,ok,Xiriirka kubbadda cagta Bariga iyo Bartamaha A...


In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
from datetime import datetime
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

# =========================
# SETTINGS
# =========================

input_file = "goobjoog_ciyaaraha_links.xlsx"
output_file = "goobjoog_ciyaaraha_scraped_articles_FIXED.xlsx"

TEST_LIMIT = None      # None = scrape all links
SAVE_EVERY = 5         # Save checkpoint every 5 articles
REQUEST_DELAY = 2.5    # Delay between requests

CATEGORY = "ciyaaro"
SOURCE = "Goobjoog"


# =========================
# SESSION
# =========================

def get_robust_session():
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False
    )

    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "so,en-US;q=0.9,en;q=0.8",
        "Connection": "keep-alive",
    })

    return session


# =========================
# SCRAPER FUNCTIONS
# =========================

def extract_headline(soup):
    headline_tag = (
        soup.select_one("h1.entry-title")
        or soup.select_one("h1.post-title")
        or soup.find("h1")
    )

    if headline_tag:
        return headline_tag.get_text(" ", strip=True)

    return ""


def extract_body(soup):
    """
    No cleaning here.
    This only extracts paragraph text from the best available article container.
    If no article container is found, it falls back to all <p> tags.
    """

    article_div = (
        soup.select_one("div.entry-content")
        or soup.select_one("article div.entry-content")
        or soup.select_one("div.post-content")
        or soup.select_one("div.article-content")
        or soup.select_one("div.td-post-content")
        or soup.find("article")
        or soup.find("main")
    )

    if article_div:
        paragraphs = article_div.find_all("p")
    else:
        paragraphs = soup.find_all("p")

    body = "\n".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
        if p.get_text(" ", strip=True)
    )

    return body


def scrape_goobjoog_article(url, session):
    try:
        response = session.get(url, timeout=25)

        if response.status_code != 200:
            return {
                "headline": "",
                "body": "",
                "status": f"failed_status_{response.status_code}",
                "error": f"HTTP status {response.status_code}"
            }

        soup = BeautifulSoup(response.content, "html.parser")

        headline = extract_headline(soup)
        body = extract_body(soup)

        if headline and body:
            status = "ok"
        elif headline and not body:
            status = "headline_only_no_body"
        elif body and not headline:
            status = "body_only_no_headline"
        else:
            status = "no_headline_no_body"

        return {
            "headline": headline,
            "body": body,
            "status": status,
            "error": ""
        }

    except Exception as e:
        return {
            "headline": "",
            "body": "",
            "status": "exception",
            "error": str(e)
        }


# =========================
# LOAD LINKS
# =========================

if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"{input_file} not found. Put it in the same folder as this notebook/script."
    )

df_links = pd.read_excel(input_file)

print("Columns found in links file:")
print(df_links.columns.tolist())

if "URL" in df_links.columns:
    url_col = "URL"
elif "url" in df_links.columns:
    url_col = "url"
else:
    raise ValueError("No URL column found. Your file must contain either 'URL' or 'url'.")

urls_to_scrape = (
    df_links[url_col]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

if TEST_LIMIT is not None:
    urls_to_scrape = urls_to_scrape[:TEST_LIMIT]

print(f"\nTotal URLs selected for scraping: {len(urls_to_scrape)}")


# =========================
# CHECKPOINT / RESUME
# =========================

scraped_data = []
processed_urls = set()

if os.path.exists(output_file):
    df_existing = pd.read_excel(output_file)

    if not df_existing.empty and "url" in df_existing.columns:
        scraped_data = df_existing.to_dict("records")
        processed_urls = set(df_existing["url"].astype(str).tolist())

        print(f"\nResuming from checkpoint: {len(processed_urls)} URLs already scraped.")
    else:
        print("\nExisting output file found, but it is empty or invalid. Starting fresh.")


# =========================
# MAIN LOOP
# =========================

session = get_robust_session()

print("\nStarting full Goobjoog Ciyaaro scrape...\n")

for i, url in enumerate(urls_to_scrape, start=1):
    if url in processed_urls:
        print(f"[{i}/{len(urls_to_scrape)}] Skipping already scraped: {url}")
        continue

    print("=" * 90)
    print(f"[{i}/{len(urls_to_scrape)}] Scraping: {url}")

    result = scrape_goobjoog_article(url, session)

    headline = result["headline"]
    body = result["body"]
    status = result["status"]
    error = result["error"]
    body_word_count = len(body.split()) if body else 0

    row = {
        "url": url,
        "headline": headline,
        "body": body,
        "category": CATEGORY,
        "source": SOURCE,
        "status": status,
        "body_word_count": body_word_count,
        "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "error": error
    }

    scraped_data.append(row)
    processed_urls.add(url)

    print(f"Status: {status}")
    print(f"Headline: {headline[:150] if headline else 'NO HEADLINE'}")
    print(f"Body words: {body_word_count}")
    print(f"Body preview: {body[:300].replace(chr(10), ' ') if body else 'NO BODY'}")

    if len(scraped_data) % SAVE_EVERY == 0:
        pd.DataFrame(scraped_data).to_excel(output_file, index=False)
        print(f"Checkpoint saved: {len(scraped_data)} rows -> {output_file}")

    time.sleep(REQUEST_DELAY)


# =========================
# FINAL SAVE + SUMMARY
# =========================

df_out = pd.DataFrame(scraped_data)
df_out.to_excel(output_file, index=False)

print("\n" + "=" * 90)
print(f"Full scraping complete. Saved {len(df_out)} rows to {output_file}")

print("\nStatus counts:")
print(df_out["status"].value_counts(dropna=False))

print("\nBody word count summary:")
print(df_out["body_word_count"].describe())

print("\nRows with no body or very short body:")
display(
    df_out[
        (df_out["body"].fillna("").str.strip() == "") |
        (df_out["body_word_count"] < 50)
    ][["url", "headline", "body_word_count", "status", "error"]].head(20)
)

display(df_out[["url", "headline", "body_word_count", "status", "body"]].head(10))

Columns found in links file:
['URL']

Total URLs selected for scraping: 674

Starting full Goobjoog Ciyaaro scrape...

[1/674] Scraping: https://goobjoog.com/2020/12/31/yaa-guulaysan-doono-gobolka-banaadir-iyo-galmudug-tartanka-maamul-goboleedyada-2020/
Status: ok
Headline: Yaa Guulaysan Doono Gobolka Banaadir Iyo Galmudug Tartanka Maamul Goboleedyada 2020
Body words: 224
Body preview: Kooxaha Kubbadda Cagta Gobolka Banaadir iyo Galmudug ayaa markii ugu horeesay wada dheeli doono heerka ugu danbeeya tartanka kubbadda cagta Maamul Goboleedyada iyo Gobolka Banaadir. Labada koox oo ka soo wada baxay Groupka A,ayaa soo tiigsaday fiinalaha tartan markii kowaad,tan iyo markii labilaaway
[2/674] Scraping: https://goobjoog.com/2020/10/25/xildhibaan-shaacir-guul-ayaan-u-rajaynaayaa-wasiir-xamsa-iyo-wasaarada-ciyaaraha-dalka/
Status: ok
Headline: Xildhibaan Shaacir “Guul Ayaan U Rajaynaayaa Wasiir Xamsa Iyo Wasaarada Ciyaaraha Dalka”
Body words: 179
Body preview: Xildhibaan Cabdullahi Maxamed Aa

,url,headline,body_word_count,status,error
14,https://goobjoog.com/2020/06/13/yuusuf-yuusuf-...,Yuusuf Yuusuf “ Wasiir Khadiijo Qalad Ayay Ku...,49,ok,
20,https://goobjoog.com/2018/11/14/natiijo-maxay-...,Natiijo:-Maxay Kusoo Dhmaatay Ciyaartii U Dhax...,0,headline_only_no_body,
40,https://goobjoog.com/2020/09/29/buurane-waa-ce...,Buurane “Waa Ceyb In Loo Gacan Qaadaa Garsoora...,41,ok,
64,https://goobjoog.com/2015/01/12/warbixinta-gan...,Warbixinta Ganacsiga ee Sanadkii 2014 (AKHRISO),39,ok,
66,https://goobjoog.com/2020/06/28/guddoomiye-saa...,Guddoomiye Saacid “Degmada Warta Nabadda Taari...,47,ok,
70,https://goobjoog.com/2020/06/16/ibraahim-geedo...,Ibraahim Geedow “Dhismaha Stadium Muqdisho Wa...,30,ok,
73,https://goobjoog.com/2019/03/28/jubaland-maxaa...,Jubaland-Maxaa Sababay Guuladaradii Kooxda Kaneva,33,ok,
79,https://goobjoog.com/2022/02/08/shirkadda-nike...,Shirkadda Nike Oo Joojisay Wadashaqeyntii Ay L...,2,ok,
80,https://goobjoog.com/2019/03/21/garsoorayaal-s...,Garsoorayaal Soomaaliyeed Oo Kasoo Muuqday Is ...,0,headline_only_no_body,
81,https://goobjoog.com/2022/02/09/garoonka-camp-...,"Garoonka Camp Nou Oo Magaca Laga Beddelayo, Sh...",2,ok,


,url,headline,body_word_count,status,body
0,https://goobjoog.com/2020/12/31/yaa-guulaysan-...,Yaa Guulaysan Doono Gobolka Banaadir Iyo Galmu...,224,ok,Kooxaha Kubbadda Cagta Gobolka Banaadir iyo Ga...
1,https://goobjoog.com/2020/10/25/xildhibaan-sha...,Xildhibaan Shaacir “Guul Ayaan U Rajaynaayaa W...,179,ok,Xildhibaan Cabdullahi Maxamed Aadan (Shaacir) ...
2,https://goobjoog.com/2024/01/21/akhriso-tartan...,Akhriso: Tartanka Dowlad Goboleedyada oo Dib u...,319,ok,Garoonka Kubadda Cagta ee Stadium Muqdisho wax...
3,https://goobjoog.com/2020/07/11/maxamuud-maxam...,Maxamuud Maxamed “Waxaan Diyaar U Nahay Horoma...,199,ok,Maamulka Ciyaaraha Degmada Diinsoor ayaa tarta...
4,https://goobjoog.com/2019/06/10/wax-badan-ka-o...,Wax Badan Ka Ogaw Kulanka Horseed VS Muqdisho ...,320,ok,Kooxda Ciidanka xooda dalka ee Horseed ayaa wa...
5,https://goobjoog.com/2020/06/16/ibraahim-geedo...,Ibraahim Geedow “Xiriirka Kubbbadda CagtaS Soo...,57,ok,Ibraahim Geedoow Cawaale oo ah qoraa iyo falan...
6,https://goobjoog.com/2020/08/29/daadir-amiin-w...,Daadir Amiin “Waan Ku Faraxsanahay Hanashada H...,110,ok,Kooxda Geeska Afrika ayaa ku guulaystay horyaa...
7,https://goobjoog.com/2019/03/28/hordhaca-horya...,Hordhaca Horyaalka Somaaliya-Yaa Guulo Badan K...,523,ok,Kooxda ciidan booliska Soomaaliyeed ee Heegan ...
8,https://goobjoog.com/2019/11/21/daaru-tarbiya-...,Daaru Tarbiya Oo Ku Guulaystay Tartanka Iskuulada,225,ok,Waxaa lasoo gaba-gabeeyay koobkii kubbadda cag...
9,https://goobjoog.com/2019/09/28/guddoomiye-cab...,Guddoomiye Cabdiqani Saciid “Xulka Dhallinyara...,337,ok,Xiriirka kubbadda cagta Bariga iyo Bartamaha A...
